# 12 — Select: SERVE ranks, EXPLORE learns

> **Module name.** This notebook exports **`selection.py`**, not `select.py`. `select` is a
> **C built-in** module and `BuiltinImporter` precedes `PathFinder` on `sys.meta_path`, so
> `import select` resolves to the standard library whatever the working directory —
> an earlier version of this notebook wrote a `select.py` that could never be imported by
> anything.

**What this is.** The step that picks which of the screened candidates to act on — and it
does two completely different jobs depending on who is asking.

**Why it matters here.** A plant engineer with an order needs a mix that is safe to cast
today. A researcher with an experiment budget needs the mixes that will teach the most,
which are by definition the ones the models know least about. Same candidate list, opposite
requirement.

**What breaks if it is ignored.** One of them silently gets the other's answer — and the
dangerous direction is the plant engineer receiving a deliberately-uncertain research mix.

---

## The two doors

| | **SERVE** | **EXPLORE** |
|---|---|---|
| who | plant engineer with an order | researcher with a budget |
| training support | **hard reject outside** | permitted, and **flagged** on every row |
| how it picks | rank on cost and carbon, subject to P(pass) thresholds | acquisition function, batch of *k* |
| standards constraints | **identical** | **identical** |
| output | mixes to cast today | mixes to learn from |
| writes to models | never | never — notebook 13 does that |

**Standards constraints do not relax in EXPLORE.** Exploration widens where the models may
be asked to predict. It never widens what IS 456 permits.

## What may not be ranked

`[PC]` A candidate whose `P_strength` is `NaN` carries `strength_status = NOT_VERIFIED`.
It **cannot be ranked**. The project's fail-safe rule says reject or revert to the verified
deterministic procedure — it does not say substitute an optimistic number. `rank_serve`
raises rather than sorting around a NaN.

## On the acquisition function

`[BO]` Bayesian optimisation is a **project research method**, not an Indian Standard
requirement. The acquisition implemented here is a documented, simple one. **No claim is
made that it beats qNEHVI, qEI or any alternative** — that would need a comparison this
project has not run, and with five independent design families such a comparison would not
be conclusive either way.

In [ ]:
# EXPORT: skip -- notebook preamble only
# Always load the CURRENT project modules from disk. Python imports a module once per
# kernel, so a kernel that was open before an earlier notebook rewrote layer0.py /
# screen.py / ... would silently keep the OLD code. Dropping them first makes every
# Run All pick up the latest files, with or without a kernel restart.
import sys as _sys
for _m in ("is_rules", "layer0", "workability_residual", "strength_residual",
           "screen", "selection", "loop"):
    _sys.modules.pop(_m, None)

import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

import is_rules as IS
import layer0 as L0
import screen as SC

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

## 1. Price and carbon — every value carries its status

`[PC]` A portfolio ranked on placeholder numbers that does not say so is the same failure
class as a compliance check that prints PASS without evaluating. Cost and carbon each
report the **worst** status among their constituents.

In [ ]:
class ValueStatus:
    """[PC] Where a rate or factor came from. Printed on every ranked output."""

    MEASURED = "measured"        # from this plant
    PUBLISHED = "published"      # a citable Indian source or EPD
    PLACEHOLDER = "placeholder"  # neither -- must be surfaced on every output


# [PC] PLACEHOLDER rates. These are NOT measured and NOT published. They exist so the
#      ranking machinery can be tested; any portfolio ranked on them says so on its face.
#      Replace with plant rates (MEASURED) or an Indian EPD (PUBLISHED) before use.
RATES_INR_PER_KG = {
    "cement": (8.00, ValueStatus.PLACEHOLDER),
    "flyash": (2.50, ValueStatus.PLACEHOLDER),
    "ggbs": (4.50, ValueStatus.PLACEHOLDER),
    "water": (0.05, ValueStatus.PLACEHOLDER),
    "admixture": (95.00, ValueStatus.PLACEHOLDER),
    "coarse_agg": (1.20, ValueStatus.PLACEHOLDER),
    "fine_agg": (1.60, ValueStatus.PLACEHOLDER),
}

CARBON_KG_PER_KG = {
    "cement": (0.90, ValueStatus.PLACEHOLDER),
    "flyash": (0.02, ValueStatus.PLACEHOLDER),
    "ggbs": (0.08, ValueStatus.PLACEHOLDER),
    "water": (0.00, ValueStatus.PLACEHOLDER),
    "admixture": (1.50, ValueStatus.PLACEHOLDER),
    "coarse_agg": (0.01, ValueStatus.PLACEHOLDER),
    "fine_agg": (0.01, ValueStatus.PLACEHOLDER),
}

_COLS = {"cement": "cement_kg_m3", "flyash": "flyash_kg_m3", "ggbs": "ggbs_kg_m3",
         "water": "water_kg_m3", "admixture": "admixture_kg_m3",
         "coarse_agg": "coarse_agg_kg_m3", "fine_agg": "fine_agg_kg_m3"}

_WORST = [ValueStatus.MEASURED, ValueStatus.PUBLISHED, ValueStatus.PLACEHOLDER]


def price_and_carbon(df, rates=None, carbon=None):
    """Cost and carbon per m3, each carrying the worst status of its constituents."""
    rates = rates or RATES_INR_PER_KG
    carbon = carbon or CARBON_KG_PER_KG
    out = df.copy()

    cost = np.zeros(len(out))
    co2 = np.zeros(len(out))
    cost_status, co2_status = ValueStatus.MEASURED, ValueStatus.MEASURED
    for key, col in _COLS.items():
        if col not in out:
            continue
        r, rs = rates[key]
        c, cs = carbon[key]
        cost += out[col].to_numpy(float) * r
        co2 += out[col].to_numpy(float) * c
        cost_status = max(cost_status, rs, key=_WORST.index)
        co2_status = max(co2_status, cs, key=_WORST.index)

    out["cost_inr_m3"] = cost
    out["carbon_kg_m3"] = co2
    out["cost_status"] = cost_status
    out["carbon_status"] = co2_status
    return out

## 2. SERVE — rank, do not chase

`[PC]` Feasibility first, then rank. A candidate is feasible when **all** of:

- `gate_decided` is True — `[IS]` every durability check decided, none hard-unverified
- inside training support — the model has seen something like it
- `P_strength` ≥ threshold and not NaN
- `P_slump` ≥ threshold

Then rank on cost and carbon. `[PC]` The default is a Pareto front, not a weighted sum: a
weighted sum invents an exchange rate between rupees and kilograms of CO₂ that nobody in
this project has agreed.

In [ ]:
def pareto_front(df, objectives=("cost_inr_m3", "carbon_kg_m3"), minimise=True):
    """[PC] Non-dominated rows. No weights, because no exchange rate has been agreed."""
    X = df[list(objectives)].to_numpy(float)
    if not minimise:
        X = -X
    keep = np.ones(len(X), dtype=bool)
    for i in range(len(X)):
        if not keep[i]:
            continue
        dominated = np.all(X <= X[i], axis=1) & np.any(X < X[i], axis=1)
        if dominated.any():
            keep[i] = False
    return df[keep].copy()


def rank_serve(cands, *, p_strength_min=0.90, p_slump_min=0.80, rates=None, carbon=None,
               front_only=True):
    """[PC] SERVE: feasible, code-decided, inside support, then ranked on cost and carbon.

    Raises rather than ranking around a NaN P_strength. A candidate whose strength
    cannot be predicted is NOT_VERIFIED, and the project's fail-safe rule forbids
    substituting an optimistic number for it.
    """
    if cands.empty:
        return cands.copy(), {"n_in": 0, "n_feasible": 0, "reason": "no candidates"}

    if "P_strength" in cands and cands["P_strength"].isna().any():
        n = int(cands["P_strength"].isna().sum())
        raise ValueError(
            f"{n} of {len(cands)} candidates have P_strength = NaN "
            f"(strength_status: {sorted(set(cands.get('strength_status', ['?'])))}). "
            f"A NOT_VERIFIED hard check may not be ranked around. Load a strength model, "
            f"or use the verified deterministic procedure, or refuse the order."
        )

    df = price_and_carbon(cands, rates, carbon)
    reasons = {}
    m = df["gate_decided"].astype(bool)
    reasons["gate not decided"] = int((~m).sum())
    if "inside_training_range" in df:
        m2 = df["inside_training_range"] != False
        reasons["outside training support"] = int((~m2).sum())
        m &= m2
    m3 = df["P_strength"] >= p_strength_min
    reasons[f"P_strength < {p_strength_min}"] = int((~m3).sum())
    m &= m3
    m4 = df["P_slump"] >= p_slump_min
    reasons[f"P_slump < {p_slump_min}"] = int((~m4).sum())
    m &= m4

    feasible = df[m].copy()
    meta = {"n_in": len(df), "n_feasible": len(feasible), "excluded": reasons,
            "cost_status": df["cost_status"].iloc[0],
            "carbon_status": df["carbon_status"].iloc[0],
            "thresholds": {"p_strength_min": p_strength_min, "p_slump_min": p_slump_min}}
    if feasible.empty:
        return feasible, meta

    ranked = pareto_front(feasible) if front_only else feasible
    ranked = ranked.sort_values(["cost_inr_m3", "carbon_kg_m3"]).reset_index(drop=True)
    meta["n_ranked"] = len(ranked)
    return ranked, meta

## 3. EXPLORE — choose the trials, not the optimum

`[BO]` The optimiser's job is **not** to find the surrogate's peak. The surrogate is built
on five design families and its out-of-group skill is negative; its peak is not information.
The job is to choose which physical trials to cast so the next model is better.

`[BO]` The acquisition below scores a candidate on predictive **uncertainty**, penalised
for being far outside the code-feasible region, and a batch is built greedily with a
**diversity penalty** so the *k* trials are not four copies of one mix.

`[PC]` The batch also carries a **joint batch risk**: if all *k* trials share one
assumption and that assumption is wrong, the whole batch is wasted. That number is printed,
not hidden.

In [ ]:
def acquisition_uncertainty(df, *, sd_col="slump_sd", feas_cols=("P_strength", "P_slump"),
                            feas_floor=0.05):
    """[BO] Score = predictive sd, weighted by how feasible the point still is.

    Deliberately simple and fully documented. NO CLAIM is made that this beats qNEHVI,
    qEI or any alternative; that needs a comparison this project has not run, and with
    five independent design families it would not be conclusive either way.

    A point with near-zero feasibility is not informative about a mix anyone would cast,
    so it is down-weighted rather than chased.
    """
    sd = df[sd_col].to_numpy(float)
    w = np.ones(len(df))
    for c in feas_cols:
        if c in df:
            w = w * np.clip(np.nan_to_num(df[c].to_numpy(float), nan=feas_floor),
                            feas_floor, 1.0)
    return sd * w


def propose_batch(cands, k=4, *, design_cols=("dv_binder_kg_m3", "dv_flyash_frac",
                                              "dv_ggbs_frac", "dv_sp_pct_binder"),
                  diversity_weight=1.0, sd_col="slump_sd"):
    """[BO] Greedy batch with a diversity penalty, so k trials are not k copies.

    Returns (batch, meta). meta carries the joint batch risk.
    """
    if cands.empty:
        return cands.copy(), {"k": 0, "reason": "no candidates"}
    df = cands.copy().reset_index(drop=True)
    score = acquisition_uncertainty(df, sd_col=sd_col)

    cols = [c for c in design_cols if c in df]
    X = df[cols].to_numpy(float)
    rng = X.max(axis=0) - X.min(axis=0)
    rng[rng == 0] = 1.0
    Xn = (X - X.min(axis=0)) / rng

    chosen = []
    for _ in range(min(k, len(df))):
        s = score.copy()
        if chosen:
            d = np.min(np.linalg.norm(Xn[:, None, :] - Xn[chosen][None, :, :], axis=2), axis=1)
            s = s * (1.0 - np.exp(-diversity_weight * d))   # penalise near-duplicates
        s[chosen] = -np.inf
        chosen.append(int(np.argmax(s)))

    batch = df.iloc[chosen].copy().reset_index(drop=True)
    batch["trial_rank"] = range(1, len(batch) + 1)
    batch["acquisition"] = "uncertainty x feasibility, greedy with diversity penalty [BO]"

    # [PC] joint batch risk -- how much of the batch dies if one shared assumption is wrong
    shared = {}
    for c in cols:
        v = batch[c].nunique()
        shared[c] = {"distinct_values": int(v), "all_same": bool(v == 1)}
    n_same = sum(1 for c in shared if shared[c]["all_same"])
    meta = {
        "k": len(batch),
        "shared_design_values": shared,
        "joint_batch_risk": (
            f"{n_same} of {len(cols)} design variables take a single value across the whole "
            f"batch. If that value rests on a wrong assumption, that fraction of the batch "
            f"is wasted together, not independently."
        ),
        "outside_support": (int((batch.get("inside_training_range") == False).sum())
                            if "inside_training_range" in batch else None),
    }
    return batch, meta

## 4. The lab package for one order — 5 designs, 3 IS trial batches, 3 exploration proposals

`[PC]` The project hands back **5 designs** and **3 physical trials** per order (the same
shape FinalSDL was built for). The 3 trials are not invented by the optimiser: they are
**IS 10262 cl 5.8 Trial Mixes No. 1/2, 3 and 4** around design 1 — same water, w/cm about
±10 % — which is how a laboratory establishes the strength-vs-w/c line for the materials
actually used (cl 5.1). Each trial comes with the model's prediction (so the result can be
judged against it), and a batch sheet the lab can weigh from.

`[BO]` The exploration proposals are separate and optional: the acquisition picks points
the models know least about, inside a trust region around the data, on the plant's own
materials and through the same IS gate.

In [ ]:
from dataclasses import replace as _replace

import workability_residual as WR

N_DESIGNS = 5        # [PC] designs handed back per order
N_EXPLORE = 3        # [PC] optional exploration proposals per order
N_CUBES = 6          # [IS] IS 516 (Part 1/Sec 1) cl 3.3 + 3.4: 3 cubes at 7 d + 3 at 28 d


def pick_designs(ranked, n=N_DESIGNS):
    """Pareto-front mixes first (cost vs carbon), then the cheapest of the rest. The models
    see total SCM, not the fly ash / GGBS split, so one per (binder, SCM, dose)."""
    if ranked is None or ranked.empty:
        return pd.DataFrame()
    r = ranked.copy()
    r["on_pareto_front"] = r.index.isin(set(pareto_front(r).index))
    d = (r.sort_values(["on_pareto_front", "cost_inr_m3"], ascending=[False, True])
          .drop_duplicates(["binder_kg_m3", "scm_frac", "sp_pct_binder"])
          .head(n).reset_index(drop=True))
    d.insert(0, "design_rank", range(1, len(d) + 1))
    d["validation_level"] = L0.ValidationLevel.COMPUTATIONALLY_COMPLIANT.name
    return d


def lab_package(req, state, *, wmodel, smodel, support, order_id,
                p_strength_min=0.90, p_slump_min=0.80, n_designs=N_DESIGNS,
                n_explore=N_EXPLORE, n_cubes=N_CUBES, sand_moisture_pct=None,
                ca_moisture_pct=None):
    """Everything a lab needs for one order. Never claims a result.

    Returns dict: status, designs, trials, batches, explore, notes.
    """
    notes = []
    serve_c, serve_meta = SC.screen(req, state, wmodel=wmodel, smodel=smodel,
                                    support=support, mode="serve")
    expl_c, _ = SC.screen(req, state, wmodel=wmodel, smodel=smodel,
                          support=support, mode="explore")
    ranked, rmeta = pd.DataFrame(), {}
    if len(serve_c):
        try:
            ranked, rmeta = rank_serve(serve_c, p_strength_min=p_strength_min,
                                       p_slump_min=p_slump_min, front_only=False)
        except ValueError as e:
            notes.append(f"REFUSED TO RANK: {e}")
    designs = pick_designs(ranked, n_designs)
    if len(designs):
        designs = designs.assign(order_id=order_id)

    # ---- the anchor for the IS 10262 cl 5.8 trial set -------------------------
    if len(designs):
        anchor, basis = designs.iloc[0], "design 1 (meets both thresholds)"
        status = "RANKED"
    elif len(serve_c) and serve_c["P_strength"].notna().any():
        anchor = serve_c.sort_values("P_strength", ascending=False).iloc[0]
        basis = (f"NO design met P_strength >= {p_strength_min} and P_slump >= {p_slump_min}. "
                 f"The trials bracket the in-support candidate with the highest P_strength "
                 f"({anchor['P_strength']:.2f}) to measure this plant's strength-vs-w/c line "
                 f"(IS 10262 cl 5.1, 5.8) -- they are learning trials, not a supply mix")
        status = "NO FEASIBLE DESIGN -- learning trials only"
    else:
        anchor, basis = None, "no in-support candidate: no trial set"
        status = "NO SERVE CANDIDATE"
    notes.append(f"trial anchor: {basis}")

    trials, batches = pd.DataFrame(), pd.DataFrame()
    if anchor is not None:
        wfit = wmodel["water_reduction_fit"]
        req_c = _replace(req, water_reduction_pct=float(
            WR.water_reduction_pct(float(anchor["dv_sp_pct_binder"]), wfit)))
        ts = L0.trial_set(req_c, state, binder_kg_m3=float(anchor["dv_binder_kg_m3"]),
                          flyash_frac=float(anchor["dv_flyash_frac"]),
                          ggbs_frac=float(anchor["dv_ggbs_frac"]),
                          admixture_pct_of_binder=float(anchor["dv_sp_pct_binder"]))
        ok = [t for t in ts if t["mix"] is not None]
        scored, _ = SC.screen(req, state, wmodel=wmodel, smodel=smodel, support=support,
                              mode="score", mixes=[t["mix"] for t in ok])
        if len(scored) != len(ok):
            raise RuntimeError("a trial mix was refused by the gate after trial_set built it")
        trials = scored.assign(
            order_id=order_id,
            trial_no=[t["trial_no"] for t in ok],
            trial_role=[t["role"] for t in ok],
            wcm_target=[t["wcm_target"] for t in ok],
            trial_note=[t["note"] for t in ok],
            anchor_basis=basis)
        for t in ts:
            if t["mix"] is None:
                notes.append(f"trial {t['trial_no']}: {t['note']}")
        vol = IS.trial_batch_volume(n_cubes, msa_mm=state.msa_mm)
        sm = sand_moisture_pct if sand_moisture_pct is not None else 0.0
        cm = ca_moisture_pct if ca_moisture_pct is not None else 0.0
        rows = []
        for t in ok:
            bs = L0.batch_sheet(t["mix"], state, volume_m3=vol["batch_m3"],
                                sand_moisture_pct=sm, ca_moisture_pct=cm)
            for r in bs["rows"]:
                rows.append({"order_id": order_id, "trial_no": t["trial_no"], **r,
                             "batch_m3": vol["batch_m3"],
                             "sand_moisture_pct_used": sm, "ca_moisture_pct_used": cm,
                             "sand_absorption_pct": state.sand_absorption_pct,
                             "ca_absorption_pct": state.ca_absorption_pct})
        batches = pd.DataFrame(rows)
        notes.append(f"batch volume {vol['batch_m3']:.3f} m3 = {n_cubes} cubes + slump + density "
                     f"+ {vol['waste_frac_pc']:.0%} margin [PC]; IS 1199 Pt 1 strength sample "
                     f">= 0.02 m3")

    explore, bmeta = pd.DataFrame(), {}
    # an exploration proposal must teach something NEW: drop the designs already handed
    # back and the trial anchor
    key = ["dv_binder_kg_m3", "dv_flyash_frac", "dv_ggbs_frac", "dv_sp_pct_binder"]
    seen = pd.concat([d[key] for d in (designs, pd.DataFrame([anchor]) if anchor is not None
                                       else pd.DataFrame()) if len(d)], ignore_index=True) \
        if (len(designs) or anchor is not None) else pd.DataFrame(columns=key)
    if len(expl_c) and len(seen):
        k_e = expl_c[key].round(6).astype(str).agg("|".join, axis=1)
        k_s = set(seen[key].round(6).astype(str).agg("|".join, axis=1))
        expl_c = expl_c[~k_e.isin(k_s)].reset_index(drop=True)
    if len(expl_c):
        explore, bmeta = propose_batch(expl_c, k=n_explore)
        explore = explore.assign(order_id=order_id,
                                 note="optional exploration proposal -- expert review before casting")
    return {"status": status, "designs": designs, "trials": trials, "batches": batches,
            "explore": explore, "explore_meta": bmeta, "notes": notes,
            "serve_meta": serve_meta, "rank_meta": rmeta,
            "best_P_strength": (float(serve_c["P_strength"].max()) if len(serve_c) else float("nan")),
            "n_serve": len(serve_c), "n_explore_candidates": len(expl_c)}

## 5. Run it for the M30 order — the plant's own materials

In [ ]:
# EXPORT: skip -- worked example
wmodel = joblib.load("workability_residual_model.joblib")
smodel = SC.load_strength_model()

# The two inputs the dataset does not record -- declared, never hidden:
FLYASH_SG = 2.20                      # ASSUMED: IS 10262 Annex B-2(d) worked-example value
ADMIXTURE_RANGE_PCT = (0.621, 1.476)  # PLANT-OBSERVED; replace with the PCE datasheet (IS 9103 cl 10.2)

ORDERS = SC.plant_orders(flyash_sg=FLYASH_SG, admixture_range_pct=ADMIXTURE_RANGE_PCT)
fam = next(o for o in ORDERS if "req" in o and o["req"].grade == "M30")
state = fam["state"]
# The order: M30, 120 mm slump, severe exposure, RCC, pumped.
req = L0.Requirement(grade="M30", target_slump_mm=120.0, exposure=IS.Exposure.SEVERE,
                     use=IS.UseClass.RCC, pumpable=True, pumpable_reduction_frac=1.0,
                     concrete_temp_c=fam["req"].concrete_temp_c)

train = pd.read_csv("04_workability_dataset.csv")
train = train[train["design_grp"].astype(str).str.contains("[A-Za-z]", na=False)]
SUPPORT = SC.training_support(train, wmodel["features"])

print(f"materials : {fam['family']} ({fam['plant']['plant_lab_id']}), {state.cement_type}, "
      f"SCMs {state.scms_available}, sand 600 um {state.sand_passing_600um}")
print(f"cl 5.1    : {state.combination_suitability_evidence}")

PKG = lab_package(req, state, wmodel=wmodel, smodel=smodel, support=SUPPORT, order_id="M30-120-severe",
                  sand_moisture_pct=fam["plant"]["sand_moisture_median"],
                  ca_moisture_pct=fam["plant"]["ca_moisture_median"])
designs, trials, batches, explore = PKG["designs"], PKG["trials"], PKG["batches"], PKG["explore"]

print(f"\nSTATUS: {PKG['status']}   (serve candidates {PKG['n_serve']})")
for n in PKG["notes"]:
    print("  - " + n)
if len(designs):
    print("\n5 DESIGNS (SERVE)")
    print(designs[["design_rank", "binder_kg_m3", "cement_kg_m3", "flyash_kg_m3", "water_kg_m3",
                   "sp_pct_binder", "wc_counted", "P_strength", "P_slump", "cost_inr_m3",
                   "on_pareto_front"]].to_string(index=False, float_format=lambda x: f"{x:8.3f}"))
if len(trials):
    print("\n3 PHYSICAL TRIALS -- IS 10262 cl 5.8 around the anchor (same water, w/cm +/-10 %)")
    print(trials[["trial_no", "binder_kg_m3", "cement_kg_m3", "flyash_kg_m3", "water_kg_m3",
                  "admixture_kg_m3", "wb_ratio", "wc_counted", "fc_pred", "fc_sd", "slump_pred",
                  "slump_sd"]].to_string(index=False, float_format=lambda x: f"{x:8.3f}"))
    for _, r in trials.iterrows():
        print(f"   trial {r.trial_no}: {r.trial_note}")
if len(explore):
    print("\nOPTIONAL EXPLORATION PROPOSALS (trust region, plant materials)")
    print(explore[["dv_binder_kg_m3", "dv_flyash_frac", "dv_sp_pct_binder", "water_kg_m3",
                   "slump_pred", "slump_sd", "fc_pred", "inside_training_range"]]
          .to_string(index=False, float_format=lambda x: f"{x:8.3f}"))
print("\nEvery row is COMPUTATIONALLY COMPLIANT and nothing more. None has been cast.")

In [ ]:
# EXPORT: skip -- artefacts
def _write(df, path, what):
    if len(df):
        df.to_csv(path, index=False)
        print(f"wrote {path:28s} -- {len(df)} {what}")
    else:
        Path(path).unlink(missing_ok=True)
        print(f"no {what} -- {path} not written")

_write(designs, "12_next_designs.csv", "designs (SERVE)")
_write(trials, "12_next_experiments.csv", "IS 10262 cl 5.8 trial batches")
_write(batches, "12_trial_batches.csv", "batch-sheet rows")
_write(explore, "12_explore_proposals.csv", "optional exploration proposals")
Path("12_select_meta.json").write_text(json.dumps(
    {"order": "M30-120-severe", "status": PKG["status"], "notes": PKG["notes"],
     "serve": {k: v for k, v in PKG["serve_meta"].items() if k != "achievability"},
     "explore_batch": PKG["explore_meta"],
     "declared_inputs": {"flyash_sg": [FLYASH_SG, "ASSUMED -- IS 10262 Annex B-2(d) example"],
                         "admixture_range_pct": [ADMIXTURE_RANGE_PCT, "PLANT-OBSERVED"]}},
    indent=2, default=str), encoding="utf-8")
print("wrote 12_select_meta.json")

## Export

In [ ]:
"""Emit selection.py from this notebook's own cells.

Same mechanism as 00_is_codes_layer.ipynb: the module is a concatenation of the
cells that were just executed, so what ships is exactly what was tested. Fixture
and export cells are excluded -- fixtures stay in the notebook where a human runs
them and reads the evidence.
"""
import json as _json
from pathlib import Path as _Path

NOTEBOOK = "12_select_serve_explore.ipynb"
MODULE   = "selection.py"

_Q = chr(34) * 3
_HEADER = "\n".join([
    _Q + "Selection - SERVE ranks feasible mixes, EXPLORE proposes trials.",
    "",
    "GENERATED FILE - do not edit. Edit {nb} and re-run its export cell.",
    "",
    "SERVE hard-rejects anything outside training support and ranks on cost and carbon.",
    "EXPLORE keeps and flags such points, because that is where a trial teaches most.",
    "Standards constraints are identical in both. Bayesian optimisation is a PROJECT",
    "RESEARCH METHOD, not an Indian Standard requirement.",
    _Q,
    "from __future__ import annotations",
    "",
    "import math",
    "from pathlib import Path",
    "",
    "import numpy as np",
    "import pandas as pd",
    "",
    "from dataclasses import replace as _replace",
    "",
    "import is_rules as IS",
    "import layer0 as L0",
    "import screen as SC",
    "import workability_residual as WR",
    "", "",
])


def export(notebook: str = NOTEBOOK, path: str = MODULE) -> None:
    nb = _json.loads(_Path(notebook).read_text(encoding="utf-8"))
    out, skipped = [], 0
    for cell in nb["cells"]:
        if cell["cell_type"] != "code":
            continue
        src = "".join(cell["source"])
        if ("def export(" in src or "# EXPORT: skip" in src
                or src.lstrip().startswith("def _t(") or src.lstrip().startswith("!")):
            skipped += 1
            continue
        lines = [ln for ln in src.split("\n")
                 if not ln.startswith(("from __future__", "import ", "from dataclasses",
                                       "from enum import", "from pathlib", "from typing",
                                       "import numpy", "import pandas"))
]
        body = "\n".join(lines).strip("\n")
        if body:
            out.append(body)

    _Path(path).write_text(_HEADER.format(nb=notebook) + "\n\n\n".join(out) + "\n",
                           encoding="utf-8")
    ns: dict = {}
    exec(compile(_Path(path).read_text(encoding="utf-8"), path, "exec"), ns)
    assert "rank_serve" in ns and "propose_batch" in ns and "pareto_front" in ns
    assert "lab_package" in ns and "pick_designs" in ns
    print(f"wrote {path}: {len(out)} cells merged, {skipped} skipped, import-checked OK")


export()

## Open items — carried, not resolved

1. `[PC]` Every rate and carbon factor is a `PLACEHOLDER`. Any portfolio ranked on them
   prints that status. They must be replaced with plant rates (`MEASURED`) or an Indian
   EPD (`PUBLISHED`) before a ranking is used for a commercial decision.
2. `[BO]` The acquisition is simple and documented, and is **not** claimed to be better
   than any alternative. A comparison against qNEHVI or qEI would need more independent
   design families than this project has.
3. `[PC]` The Pareto front carries no weights. If the plant later states an exchange rate
   between rupees and kilograms of CO₂, a scalarised ranking becomes available — and that
   rate is a **plant input**, never a modelling choice.